### LLM Call + Function

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env_rag")

api_key = os.getenv("CO_API_KEY")
print(api_key)

In [ ]:
openai = OpenAI()

#### Reading PDF and txt file.

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a useful assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
# It forgets the previous conversation
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In the OpenAI chat completions API, there are __3 standard roles__:

1. __`system`__ — Sets the assistant's behavior/persona
2. __`user`__ — Represents the human user's messages
3. __`assistant`__ — Represents the model's responses (used to pass conversation history)

There is also a __4th role__ (`tool`) used specifically when handling tool/function calls — it returns the result of a tool execution back to the model.


In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

#### Tools and Agents

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))
# display((system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

### Just created a simple function, which is doing the same thing as above

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    print("----")
    print(messages)
    print("----")
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

### And now - TOOLS!

* Let's start with a function

In [ ]:
# Function to record an email address to a file. This is a simple example of how you might implement a tool that the AI can call to perform an action.
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("ab@testy.com")

### Step 1 - write some json to describe the tool


In [ ]:
record_email_tool_json = {
    
    "name": "record_email_tool",
    
    "description": "Use this tool to record that a user provided their email address",
    
    "parameters": {
    
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    
    }
}


In [ ]:
print((record_email_tool_json))

In [ ]:
tools = [
    {"type": "function", "function": record_email_tool_json}
]

In [ ]:
tools

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [ ]:
# system_prompt - this value is defined above in the code, and contains the system prompt that defines the AI's role and context.
def chat(message, history):
    # Build the conversation: system prompt + prior conversation history + the new user message
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # Send the conversation to the model, also passing the list of available `tools`
    # so the model can decide to call one if needed (e.g., to record an email)
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    # Check if the model decided to call a tool instead of replying directly
    if response.choices[0].finish_reason == "tool_calls":
            # Grab the assistant's message that contains the tool call request
            message = response.choices[0].message

            # Get the first (and only, in this case) tool call requested by the model
            tool_call = message.tool_calls[0]

            # Parse the JSON arguments the model generated for the tool call,
            # and extract the "email" argument value
            email = json.loads(tool_call.function.arguments).get("email")

            # Actually execute the tool locally (e.g., save/record the email somewhere)
            record_email_tool(email)

            # Add the assistant's tool-call message back into the conversation history
            messages.append(message)

            # Add a "tool" role message confirming the tool executed successfully,
            # linked to the original tool_call via tool_call_id (required by OpenAI's API)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})

            # Send the updated conversation (including the tool result) back to the model
            # so it can generate a final natural-language reply to the user
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    # Return the final text reply to be shown in the chat UI
    return response.choices[0].message.content


In [ ]:
history = []
while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break
    reply = chat(user_input, history)
    print("Bot:", reply)
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": reply})

## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
history = []
while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break
    reply = chat(user_input, history)
    print("Bot:", reply)
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": reply})